# 2 · `data_nowcast_from_images.ipynb`

Build the nowcast benchmark **directly from raw image files** (no video). Steps: down-size each image
to 64×64, drop duplicate frames, match each image to the concurrent PV value, and write an HDF5 with
`trainval` / `test` groups (same layout as the official `2017_2019_images_pv_processed.hdf5`).

**Needs in Drive:**
- `SKIPPD/images_raw/` — the extracted `{Year}_{Month}_images_raw.tar` files (any nested folders
  are fine; the notebook searches recursively). Source: https://purl.stanford.edu/sm043zf7254
- `SKIPPD/pv_processed_10s.csv` — from notebook 1.

> **Reconstruction note.** Clean, Colab-runnable reimplementation following the purpose the SKIPP'D
> README describes — not a byte-for-byte copy of the authors' notebook. **Image-only pipeline: no
> video processing.** All data lives in `/content/drive/MyDrive/SKIPPD/`.


## Mount Google Drive

Everything reads from and writes to `/content/drive/MyDrive/SKIPPD/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/SKIPPD"
os.makedirs(BASE, exist_ok=True)
print("Working folder:", BASE)
print("Contents:", os.listdir(BASE))

Mounted at /content/drive
Working folder: /content/drive/MyDrive/SKIPPD
Contents: ['2019_pv_raw.csv', 'pv_processed_10s.csv', 'images_raw', 'images_pv_processed.hdf5', 'times_test.npy', 'times_trainval.npy', 'model_output', 'forecast_samples.npz']


## Install & imports

In [2]:
!pip install -q h5py tqdm pillow
import glob, h5py
import numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm

## Timestamp parser (edit formats here if your filenames differ)

In [3]:
import re
from datetime import datetime

# Try these explicit formats first; add your own if needed.
_FORMATS = ("%Y%m%d_%H%M%S", "%Y_%m_%d_%H_%M_%S", "%Y_%m_%d_%H%M%S",
            "%Y-%m-%d_%H-%M-%S", "%Y%m%d%H%M%S")

def parse_img_time(path):
    """Return a datetime parsed from an image filename.
    Falls back to grabbing the first 14 digits as YYYYMMDDHHMMSS."""
    base = os.path.splitext(os.path.basename(path))[0]
    for fmt in _FORMATS:
        try:
            return datetime.strptime(base, fmt)
        except ValueError:
            pass
    digits = re.sub(r"\D", "", base)          # keep only digits
    if len(digits) >= 14:
        return datetime.strptime(digits[:14], "%Y%m%d%H%M%S")
    if len(digits) >= 12:
        return datetime.strptime(digits[:12], "%Y%m%d%H%M")
    raise ValueError(f"Cannot parse a timestamp from: {base}")

## Inputs / outputs (all in Drive)

In [4]:
IMAGE_DIR   = os.path.join(BASE, "images_raw")   # folder of raw .jpg images (searched recursively)
PV_CSV      = os.path.join(BASE, "pv_processed_10s.csv")
IMG_SIZE    = 64
MATCH_SEC   = 30                                  # max seconds between image and PV record
OUT_HDF5    = os.path.join(BASE, "images_pv_processed.hdf5")

pv = pd.read_csv(PV_CSV, index_col=0, parse_dates=True)["pv"].sort_index()
print("PV records:", len(pv))

PV records: 1272323


In [5]:
import os

print("IMAGE_DIR =", IMAGE_DIR)
print("exists:", os.path.isdir(IMAGE_DIR))
print("top-level contents:", os.listdir(IMAGE_DIR)[:20])

# walk the whole tree and count by extension
import collections
ext = collections.Counter()
sample = []
for root, dirs, files in os.walk(IMAGE_DIR):
    for f in files:
        ext[os.path.splitext(f)[1].lower()] += 1
        if len(sample) < 10:
            sample.append(os.path.join(root, f))

print("\nExtensions under raw_images:", dict(ext))
print("\nSample paths:")
for s in sample:
    print("  ", s)

IMAGE_DIR = /content/drive/MyDrive/SKIPPD/images_raw
exists: True
top-level contents: ['2019_01_images_raw.tar', '01']

Extensions under raw_images: {'.tar': 1, '.jpg': 3935}

Sample paths:
   /content/drive/MyDrive/SKIPPD/images_raw/2019_01_images_raw.tar
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060000.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060040.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060140.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060240.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060340.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060440.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060540.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060640.jpg
   /content/drive/MyDrive/SKIPPD/images_raw/01/26/20190126060740.jpg


In [ ]:
import tarfile, os

TAR = os.path.join(IMAGE_DIR, "2019_01_images_raw.tar")
print("Extracting", TAR, "...")

with tarfile.open(TAR) as t:
    members = t.getnames()
    print("Archive contains", len(members), "entries")
    print("First few:", members[:5])
    t.extractall(IMAGE_DIR)

print("Done.")

Extracting /content/drive/MyDrive/SKIPPD/images_raw/2019_01_images_raw.tar ...
Archive contains 3941 entries
First few: ['01', '01/24', '01/24/20190124060000.jpg', '01/24/20190124060100.jpg', '01/24/20190124060200.jpg']


/tmp/ipykernel_1127/3490751510.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(IMAGE_DIR)


In [ ]:
from datetime import datetime
sample = "01/24/20190124060000.jpg"
print(parse_img_time(sample))   # expect 2019-01-24 06:00:00

## Find image files

Searches `images_raw/` recursively for `.jpg`/`.jpeg`/`.png`, sorted by their parsed timestamp so the
sequence is chronological.

In [ ]:
exts = ("*.jpg", "*.jpeg", "*.png")
paths = []
for e in exts:
    paths += glob.glob(os.path.join(IMAGE_DIR, "**", e), recursive=True)
# keep only files we can timestamp, then sort chronologically
dated = []
for p in paths:
    try:
        dated.append((parse_img_time(p), p))
    except ValueError:
        continue
dated.sort()
print(f"Found {len(paths)} image files, {len(dated)} with a readable timestamp")

## Down-size, de-duplicate, match to PV

In [ ]:
images, pv_vals, times = [], [], []
prev = None
for t, p in tqdm(dated):
    pos = pv.index.get_indexer([pd.Timestamp(t)], method="nearest")[0]
    if abs((pv.index[pos] - pd.Timestamp(t)).total_seconds()) > MATCH_SEC:
        continue                                            # no PV close enough
    arr = np.asarray(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), np.uint8)
    if prev is not None and np.array_equal(arr, prev):
        continue                                            # repeated frame
    prev = arr
    images.append(arr); pv_vals.append(float(pv.iloc[pos])); times.append(np.datetime64(t))

images  = np.stack(images) if images else np.empty((0, IMG_SIZE, IMG_SIZE, 3), np.uint8)
pv_vals = np.array(pv_vals, np.float32)
times   = np.array(times)
print("Matched image/PV pairs:", len(images))

## Partition and write HDF5 to Drive

Last ~15% held out as test (swap in the benchmark's 20 specific test days here for an exact match).

In [ ]:
if len(images):
    split = int(len(images) * 0.85)
    with h5py.File(OUT_HDF5, "w") as f:
        g = f.create_group("trainval")
        g.create_dataset("images_log", data=images[:split], compression="gzip")
        g.create_dataset("pv_log",     data=pv_vals[:split])
        g = f.create_group("test")
        g.create_dataset("images_log", data=images[split:], compression="gzip")
        g.create_dataset("pv_log",     data=pv_vals[split:])
    np.save(os.path.join(BASE, "times_trainval.npy"), times[:split])
    np.save(os.path.join(BASE, "times_test.npy"),     times[split:])
    print("Wrote", OUT_HDF5)
    print("trainval:", split, "| test:", len(images) - split)
else:
    print("No matched pairs — check IMAGE_DIR, the filename timestamp format, and the PV file.")

Quick sanity-check preview:

In [ ]:
import matplotlib.pyplot as plt
if len(images):
    fig, ax = plt.subplots(1, 5, figsize=(14, 3))
    for i in range(min(5, len(images))):
        ax[i].imshow(images[i]); ax[i].set_title(f"{pv_vals[i]:.1f} kW"); ax[i].axis("off")
    plt.tight_layout(); plt.show()